# ML Modeling: Linear Regression Analysis for GDM Association (Placenta)

## Overview
This notebook performs linear regression analysis to determine if there is a significant difference
in metabolite levels between GDM positive and non-GDM women, while accounting for covariates.
We analyze **placenta** metabolites and calculate effect sizes and adjusted p-values.


## Step 1-2: Set Up Big Loop for Each Metabolite (Placenta Data)

We will create a loop that iterates over each metabolite. For placenta data:
- Rows = samples  
- Columns = metabolites  
- For each metabolite i: extract metabolite[i] (all rows, ith column)
- Fit model: `lm(metabolite[i] ~ GDM + covariates)`


## Setup: Load Core Inputs (Self-Contained)

Loading:
- `placenta_matrix` from `PLACENTA_DATA_T`
- Covariates and supplements from `src/raw/mata2022_caco_mar242022.xlsx`
- `Sample` IDs from the placenta mapping file (for row alignment only)
- `covariate_matrix_num` (aligned to placenta sample order)
- Output directories for Placenta-specific CSV and image files


In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

import statsmodels.formula.api as smf
import statsmodels.api as sm


In [ ]:
# Self-contained setup: load placenta metabolite matrix and build aligned covariate matrix

sys.path.insert(0, str(Path(os.getcwd()).parent.parent))
from src.utils.config import PLACENTA_DATA_T, PLACENTA_ANNO, OUTPUTS_DIR

# 1) Load placenta metabolite matrix (rows=samples, columns=metabolites)
placenta_matrix = pd.read_csv(PLACENTA_DATA_T, index_col='Sample').astype(float)
placenta_matrix.index = placenta_matrix.index.astype(str)

# 2) Load raw maternal covariate file
src_dir = Path(PLACENTA_DATA_T).resolve().parents[2]
mata_path = src_dir / 'raw' / 'mata2022_caco_mar242022.xlsx'
if not mata_path.exists():
    raise FileNotFoundError(f'Required maternal covariate file not found: {mata_path}')

mata_df = pd.read_excel(mata_path)

# 3) Load Sample ID bridge from placenta mapping output (used only for sample alignment)
gdm_map_path = os.path.join(OUTPUTS_DIR, 'pca', 'output_csv', '06_pc_scores', 'placenta_mapped_with_pc1_pc2.csv')
if not os.path.exists(gdm_map_path):
    raise FileNotFoundError(f'Required mapping file not found: {gdm_map_path}')

sample_bridge = pd.read_csv(gdm_map_path)
required_bridge_cols = ['Sample', 'IDCode']
missing_bridge = [c for c in required_bridge_cols if c not in sample_bridge.columns]
if missing_bridge:
    raise ValueError(f'Mapping file missing required columns: {missing_bridge}')

sample_bridge = sample_bridge[required_bridge_cols].copy()
sample_bridge['Sample'] = sample_bridge['Sample'].astype(str).str.strip()
sample_bridge = sample_bridge[sample_bridge['Sample'].str.len() > 0]
sample_bridge['IDCode'] = pd.to_numeric(sample_bridge['IDCode'], errors='coerce')
sample_bridge = sample_bridge.dropna(subset=['IDCode']).drop_duplicates(subset=['Sample'], keep='first')

# 4) Merge raw maternal covariates to Sample IDs via IDCode
map_df = sample_bridge.merge(mata_df, on='IDCode', how='left')
map_df = map_df.drop_duplicates(subset=['Sample'], keep='first').set_index('Sample')

# 5) Resolve required covariates by aliases
covariate_aliases = {
    'maternal_age': ['Ma_age', 'maternal_age'],
    'bmi': ['pre_BMI_new', 'BMI', 'Ma_BMI'],
    'diet_protein': ['diet_protein', 'protein', 'protein_intake'],
    'diet_fruit_vegetables': ['diet_vegfruit', 'fruit_vegetable', 'fruit_veg'],
}
required_physical_activity_col = 'O2_PHYSACTIV'

selected_covariates = {}
for label, aliases in covariate_aliases.items():
    selected = next((col for col in aliases if col in map_df.columns), None)
    if selected is not None:
        selected_covariates[label] = selected

if required_physical_activity_col not in map_df.columns:
    phys_candidates = [
        c for c in map_df.columns
        if 'phys' in str(c).lower() or 'activ' in str(c).lower()
    ]
    raise ValueError(
        f"Required physical activity column '{required_physical_activity_col}' not found. "
        f'Candidates: {phys_candidates}'
    )
selected_covariates['physical_activity'] = required_physical_activity_col

# 6) Align covariate rows to placenta matrix row order
sample_order = placenta_matrix.index
ordered_source_cols = ['GDM'] + list(selected_covariates.values())
if 'GDM' not in map_df.columns:
    raise ValueError('GDM column not found after merging maternal covariates.')

covariate_matrix_raw = map_df.reindex(sample_order)[ordered_source_cols].copy()
covariate_matrix_raw.index.name = 'Sample'

rename_map = {'GDM': 'GDM'}
rename_map.update({src: label for label, src in selected_covariates.items()})
covariate_matrix_raw = covariate_matrix_raw.rename(columns=rename_map)

# 7) Type handling: GDM and physical_activity as categorical factors; others numeric
covariate_matrix_num = covariate_matrix_raw.copy()
gdm_numeric = pd.to_numeric(covariate_matrix_num['GDM'], errors='coerce')
covariate_matrix_num['GDM'] = pd.Categorical(gdm_numeric, categories=[0, 1])

# physical_activity source coding: 1=No, 2=Yes -> model coding: 0=No, 1=Yes
phys_numeric = pd.to_numeric(covariate_matrix_num['physical_activity'], errors='coerce')
phys_observed = set(phys_numeric.dropna().unique())
if phys_observed.issubset({1, 2}):
    phys_binary = phys_numeric.map({1: 0, 2: 1})
elif phys_observed.issubset({0, 1}):
    phys_binary = phys_numeric
else:
    phys_binary = pd.Series(
        np.where(phys_numeric.isna(), np.nan, (phys_numeric > 0).astype(int)),
        index=phys_numeric.index,
    )
phys_binary = pd.array(phys_binary, dtype='Int64')
covariate_matrix_num['physical_activity'] = pd.Categorical(phys_binary, categories=[0, 1])

numeric_covariates = [c for c in covariate_matrix_num.columns if c not in ['GDM', 'physical_activity']]
covariate_matrix_num[numeric_covariates] = covariate_matrix_num[numeric_covariates].apply(pd.to_numeric, errors='coerce')

cov_matrix = covariate_matrix_num.copy()

# Auto-discover samples with missing physical_activity (used as fallback in KNN cells)
_pa_numeric = pd.to_numeric(
    covariate_matrix_raw.get('physical_activity', pd.Series(dtype=float)), errors='coerce'
)
fallback_samples = covariate_matrix_raw.index[_pa_numeric.isna()].tolist()
print(f'Samples with missing physical_activity: {fallback_samples}')

# 8) Output directories
pl_csv_dir = os.path.join(OUTPUTS_DIR, 'ml_modeling', 'Placenta', 'Python_outputs', 'CSV')
pl_img_dir = os.path.join(OUTPUTS_DIR, 'ml_modeling', 'Placenta', 'Python_outputs', 'images')
os.makedirs(pl_csv_dir, exist_ok=True)
os.makedirs(pl_img_dir, exist_ok=True)

print('Setup complete')
print('Placenta matrix shape:', placenta_matrix.shape)
print('Covariate matrix shape:', cov_matrix.shape)
print('GDM dtype:', cov_matrix['GDM'].dtype)
print('physical_activity dtype:', cov_matrix['physical_activity'].dtype)
print('physical_activity levels (0=No, 1=Yes):', cov_matrix['physical_activity'].cat.categories.tolist())
print('Resolved numeric covariates:', numeric_covariates)
print('Maternal source file:', mata_path)
print('CSV output dir:', pl_csv_dir)
print('Images output dir:', pl_img_dir)


In [ ]:
placenta_matrix.head()


In [ ]:
cov_matrix.head(40)


In [ ]:
import numpy as np

print('Missing values by covariate:')
print(covariate_matrix_num.isnull().sum())
print(f'\nTotal missing values: {covariate_matrix_num.isnull().sum().sum()}')


In [ ]:
import matplotlib.pyplot as plt

total_n = len(cov_matrix)
missing_counts = cov_matrix.isna().sum()
available_counts = total_n - missing_counts

plot_order = available_counts.sort_values(ascending=False).index.tolist()

missing_sample_labels = {}
for col in plot_order:
    missing_samples = cov_matrix.index[cov_matrix[col].isna()].tolist()
    missing_sample_labels[col] = ', '.join(missing_samples) if missing_samples else 'None'

# 1) Availability
fig, ax = plt.subplots(figsize=(10, 6))
ypos = range(len(plot_order))
ax.barh(ypos, [total_n] * len(plot_order), color='#d0d2d8', label='Total')
ax.barh(ypos, [available_counts[c] for c in plot_order], color='#3465d9', label='Available (non-missing)')
ax.set_yticks(list(ypos))
ax.set_yticklabels(plot_order)
ax.invert_yaxis()
ax.set_xlabel('Number of Samples')
ax.set_title('Covariate Availability Out of Total Samples')
ax.grid(axis='x', linestyle='--', alpha=0.3)
ax.legend(loc='lower right')
for i, c in enumerate(plot_order):
    ax.text(available_counts[c] + 0.2, i, f'{int(available_counts[c])}', va='center', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(pl_img_dir, 'covariate_availability.png'), dpi=150, bbox_inches='tight')
plt.show()

# 2) Missingness
fig, ax = plt.subplots(figsize=(13, 8))
ax.barh(ypos, [total_n] * len(plot_order), color='#d0d2d8', label='Total')
ax.barh(ypos, [missing_counts[c] for c in plot_order], color='#e5252a', label='Missing')
ax.set_yticks(list(ypos))
ax.set_yticklabels(plot_order)
ax.invert_yaxis()
ax.set_xlabel('Number of Samples')
ax.set_title('Covariate Missingness Out of Total Samples')
ax.grid(axis='x', linestyle='--', alpha=0.3)
ax.legend(loc='lower right')
for i, c in enumerate(plot_order):
    ax.text(missing_counts[c] + 0.3, i, missing_sample_labels[c], va='center', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(pl_img_dir, 'covariate_missingness.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
cov_matrix.head(40)


In [ ]:
if 'maternal_age' not in cov_matrix.columns:
    raise KeyError("'maternal_age' column not found in cov_matrix")
maternal_age_mean = pd.to_numeric(cov_matrix['maternal_age'], errors='coerce').mean()
print(f'Mean Maternal Age: {maternal_age_mean:.4f}')


In [ ]:
if 'maternal_age' not in cov_matrix.columns:
    raise KeyError("'maternal_age' column not found in cov_matrix")
maternal_age_std = pd.to_numeric(cov_matrix['maternal_age'], errors='coerce').std()
print(f'Standard Deviation of Maternal Age: {maternal_age_std:.4f}')


In [ ]:
maternal_age_series = pd.to_numeric(cov_matrix['maternal_age'], errors='coerce')
gdm_series = cov_matrix['GDM'].astype(str)

summary_rows = []
for group_name, group_values in [
    ('Non-GDM', maternal_age_series[gdm_series.isin(['0', '0.0'])]),
    ('GDM',     maternal_age_series[gdm_series.isin(['1', '1.0'])]),
    ('All',     maternal_age_series),
]:
    summary_rows.append({
        'Group': group_name,
        'Total': int(group_values.notna().sum()),
        'Mean_Maternal_Age': group_values.mean(),
        'SD_Maternal_Age':   group_values.std(),
    })

maternal_age_summary = pd.DataFrame(summary_rows)
print('Maternal Age Summary Table:')
display(maternal_age_summary)


In [ ]:
bmi_series = pd.to_numeric(cov_matrix['bmi'], errors='coerce')
gdm_series = cov_matrix['GDM'].astype(str)

bmi_rows = []
for group_name, group_values in [
    ('Non-GDM', bmi_series[gdm_series.isin(['0', '0.0'])]),
    ('GDM',     bmi_series[gdm_series.isin(['1', '1.0'])]),
    ('All',     bmi_series),
]:
    bmi_rows.append({
        'Group': group_name,
        'Total': int(group_values.notna().sum()),
        'Mean_BMI': group_values.mean(),
        'SD_BMI':   group_values.std(),
    })

bmi_summary = pd.DataFrame(bmi_rows)
print('BMI Summary Table:')
display(bmi_summary)


In [ ]:
protein_series = pd.to_numeric(cov_matrix['diet_protein'], errors='coerce')
gdm_series = cov_matrix['GDM'].astype(str)

protein_rows = []
for group_name, group_values in [
    ('Non-GDM', protein_series[gdm_series.isin(['0', '0.0'])]),
    ('GDM',     protein_series[gdm_series.isin(['1', '1.0'])]),
    ('All',     protein_series),
]:
    protein_rows.append({
        'Group': group_name,
        'Total': int(group_values.notna().sum()),
        'Sum_Protein_Frequency':  group_values.sum(),
        'Mean_Protein_Frequency': group_values.mean(),
        'SD_Protein_Frequency':   group_values.std(),
    })

protein_summary = pd.DataFrame(protein_rows)
print('Protein Frequency Summary Table:')
display(protein_summary)


In [ ]:
vegfruit_series = pd.to_numeric(cov_matrix['diet_fruit_vegetables'], errors='coerce')
gdm_series = cov_matrix['GDM'].astype(str)

vegfruit_rows = []
for group_name, group_values in [
    ('Non-GDM', vegfruit_series[gdm_series.isin(['0', '0.0'])]),
    ('GDM',     vegfruit_series[gdm_series.isin(['1', '1.0'])]),
    ('All',     vegfruit_series),
]:
    vegfruit_rows.append({
        'Group': group_name,
        'Total': int(group_values.notna().sum()),
        'Sum_VegFruits_Frequency':  group_values.sum(),
        'Mean_VegFruits_Frequency': group_values.mean(),
        'SD_VegFruits_Frequency':   group_values.std(),
    })

vegfruit_summary = pd.DataFrame(vegfruit_rows)
print('Vegetables and Fruits Frequency Summary Table:')
display(vegfruit_summary)


In [ ]:
pa_series = pd.to_numeric(cov_matrix['physical_activity'], errors='coerce')
gdm_series = cov_matrix['GDM'].astype(str)

pa_summary_rows = []
for group_name, group_mask in [
    ('Non-GDM', gdm_series.isin(['0', '0.0'])),
    ('GDM',     gdm_series.isin(['1', '1.0'])),
    ('All',     pd.Series([True] * len(cov_matrix), index=cov_matrix.index)),
]:
    group_values = pa_series[group_mask].dropna()
    total = int(group_values.shape[0])
    yes_n = int((group_values == 1).sum())
    no_n  = int((group_values == 0).sum())
    pa_summary_rows.append({
        'Group': group_name,
        'Yes_n(%)': f'{yes_n} ({(100 * yes_n / total):.1f}%)' if total else '0 (0.0%)',
        'No_n(%)':  f'{no_n} ({(100 * no_n / total):.1f}%)'  if total else '0 (0.0%)',
        'Total': total,
    })

pa_summary = pd.DataFrame(pa_summary_rows)
print('Physical Activity in Last 3 Months Summary Table:')
display(pa_summary)


In [ ]:
combined_summary_rows = []

for _, row in maternal_age_summary.iterrows():
    combined_summary_rows.append({'Metric': 'Maternal Age', 'Group': row['Group'],
        'Total': int(row['Total']), 'Mean': row['Mean_Maternal_Age'], 'SD': row['SD_Maternal_Age'],
        'Sum': pd.NA, 'Yes_n(%)': pd.NA, 'No_n(%)': pd.NA})

for _, row in bmi_summary.iterrows():
    combined_summary_rows.append({'Metric': 'BMI', 'Group': row['Group'],
        'Total': int(row['Total']), 'Mean': row['Mean_BMI'], 'SD': row['SD_BMI'],
        'Sum': pd.NA, 'Yes_n(%)': pd.NA, 'No_n(%)': pd.NA})

for _, row in protein_summary.iterrows():
    combined_summary_rows.append({'Metric': 'Protein frequency (times/week)', 'Group': row['Group'],
        'Total': int(row['Total']), 'Mean': row['Mean_Protein_Frequency'],
        'SD': row['SD_Protein_Frequency'], 'Sum': row['Sum_Protein_Frequency'],
        'Yes_n(%)': pd.NA, 'No_n(%)': pd.NA})

for _, row in vegfruit_summary.iterrows():
    combined_summary_rows.append({'Metric': 'Vegetables and fruits frequency (times/week)', 'Group': row['Group'],
        'Total': int(row['Total']), 'Mean': row['Mean_VegFruits_Frequency'],
        'SD': row['SD_VegFruits_Frequency'], 'Sum': row['Sum_VegFruits_Frequency'],
        'Yes_n(%)': pd.NA, 'No_n(%)': pd.NA})

for _, row in pa_summary.iterrows():
    combined_summary_rows.append({'Metric': 'Physical Activity in last 3 months', 'Group': row['Group'],
        'Total': int(row['Total']), 'Mean': pd.NA, 'SD': pd.NA, 'Sum': pd.NA,
        'Yes_n(%)': row['Yes_n(%)'], 'No_n(%)': row['No_n(%)']})

combined_summary = pd.DataFrame(combined_summary_rows)
combined_summary = combined_summary[['Metric', 'Group', 'Total', 'Sum', 'Mean', 'SD', 'Yes_n(%)', 'No_n(%)']]

before_path = os.path.join(pl_csv_dir, 'Table1_before_cleaning.csv')
combined_summary.to_csv(before_path, index=False)
print(f'Saved Table1 before cleaning to: {before_path}')
display(combined_summary)


In [ ]:
impute_cols = ['diet_protein', 'diet_fruit_vegetables']

before_missing = cov_matrix[impute_cols].isna().sum()
impute_values = {}

for col in impute_cols:
    mean_val = cov_matrix[col].mean(skipna=True)
    impute_values[col] = mean_val
    cov_matrix[col] = cov_matrix[col].fillna(mean_val)

after_missing = cov_matrix[impute_cols].isna().sum()

print('Imputed values used:')
for col, value in impute_values.items():
    print(f'  {col}: {value}')
print('\nMissing values before imputation:')
print(before_missing)
print('\nMissing values after imputation:')
print(after_missing)


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from IPython.display import display

target_col = 'physical_activity'
knn_impute_df = cov_matrix.copy()

target_numeric = pd.to_numeric(knn_impute_df[target_col], errors='coerce')
feature_cols = [col for col in knn_impute_df.columns if col != target_col]
feature_matrix = knn_impute_df[feature_cols].apply(pd.to_numeric, errors='coerce')
feature_matrix = feature_matrix.fillna(feature_matrix.median(numeric_only=True)).fillna(0)

observed_mask = target_numeric.notna()
missing_mask  = target_numeric.isna()
use_fallback  = int(missing_mask.sum()) == 0

if use_fallback:
    if fallback_samples:
        missing_mask  = knn_impute_df.index.isin(fallback_samples)
        observed_mask = ~missing_mask
        print('No missing physical_activity values found; reconstructing KNN table for previously imputed samples.')
    else:
        print('No missing physical_activity values and no fallback samples — skipping KNN reconstruction.')
else:
    print(f'Found {int(missing_mask.sum())} missing physical_activity value(s); imputing those rows.')

missing_mask_array  = np.asarray(missing_mask)
observed_mask_array = np.asarray(observed_mask)
missing_index  = knn_impute_df.index[missing_mask_array]
observed_index = knn_impute_df.index[observed_mask_array]

scaler = StandardScaler()
scaled_features = scaler.fit_transform(feature_matrix)

n_neighbors = min(5, int(observed_mask_array.sum()))
if n_neighbors == 0:
    raise RuntimeError('No observed samples available for KNN imputation.')

nn = NearestNeighbors(n_neighbors=n_neighbors, metric='euclidean')
nn.fit(scaled_features[observed_mask_array])
distances, neighbor_positions = nn.kneighbors(scaled_features[missing_mask_array])

imputation_rows = []
for sample_id, neighbor_pos, dist in zip(missing_index, neighbor_positions, distances):
    neighbor_samples = observed_index.to_numpy()[neighbor_pos]
    neighbor_labels  = target_numeric.loc[neighbor_samples].astype(int).to_numpy()
    weights          = 1 / (dist + 1e-8)
    weighted_vote    = float(np.average(neighbor_labels, weights=weights))
    imputed_value    = int(weighted_vote >= 0.5)

    if not use_fallback:
        knn_impute_df.loc[sample_id, target_col] = imputed_value

    imputation_rows.append({
        'sample_id': sample_id,
        'nearest_neighbors': ', '.join(neighbor_samples.tolist()),
        'neighbor_distances': [round(x, 4) for x in dist.tolist()],
        'neighbor_values': neighbor_labels.tolist(),
        'weighted_avg': round(weighted_vote, 4),
        'filled_value': imputed_value,
    })

if not use_fallback:
    cov_matrix[target_col] = pd.Categorical(
        pd.to_numeric(knn_impute_df[target_col], errors='coerce').astype('Int64'),
        categories=[0, 1],
    )

imputation_table = pd.DataFrame(imputation_rows)
print('KNN imputation summary:')
display(imputation_table)

print('\nMissing values after KNN imputation:')
if use_fallback:
    display(pd.DataFrame({'missing_count': [0]}, index=[target_col]))
else:
    display(cov_matrix[[target_col]].isna().sum().to_frame(name='missing_count'))

cov_clean = cov_matrix.copy()
print('\nCreated cleaned covariate matrix: cov_clean')


In [ ]:
combined_clean_rows = []

gdm_series_clean = cov_clean['GDM'].astype(str)

for metric, series in [
    ('Maternal Age', pd.to_numeric(cov_clean['maternal_age'], errors='coerce')),
    ('BMI',         pd.to_numeric(cov_clean['bmi'],           errors='coerce')),
]:
    for group_name, mask in [
        ('Non-GDM', gdm_series_clean.isin(['0', '0.0'])),
        ('GDM',     gdm_series_clean.isin(['1', '1.0'])),
        ('All',     pd.Series([True]*len(cov_clean), index=cov_clean.index)),
    ]:
        vals = series[mask]
        combined_clean_rows.append({
            'Metric': metric, 'Group': group_name,
            'Total': int(vals.notna().sum()), 'Sum': pd.NA,
            'Mean': vals.mean(), 'SD': vals.std(),
            'Yes_n(%)': pd.NA, 'No_n(%)': pd.NA,
        })

for metric, series in [
    ('Protein frequency (times/week)',               pd.to_numeric(cov_clean['diet_protein'],          errors='coerce')),
    ('Vegetables and fruits frequency (times/week)', pd.to_numeric(cov_clean['diet_fruit_vegetables'], errors='coerce')),
]:
    for group_name, mask in [
        ('Non-GDM', gdm_series_clean.isin(['0', '0.0'])),
        ('GDM',     gdm_series_clean.isin(['1', '1.0'])),
        ('All',     pd.Series([True]*len(cov_clean), index=cov_clean.index)),
    ]:
        vals = series[mask]
        combined_clean_rows.append({
            'Metric': metric, 'Group': group_name,
            'Total': int(vals.notna().sum()), 'Sum': vals.sum(),
            'Mean': vals.mean(), 'SD': vals.std(),
            'Yes_n(%)': pd.NA, 'No_n(%)': pd.NA,
        })

pa_series_clean = pd.to_numeric(cov_clean['physical_activity'], errors='coerce')
for group_name, mask in [
    ('Non-GDM', gdm_series_clean.isin(['0', '0.0'])),
    ('GDM',     gdm_series_clean.isin(['1', '1.0'])),
    ('All',     pd.Series([True]*len(cov_clean), index=cov_clean.index)),
]:
    vals  = pa_series_clean[mask].dropna()
    total = int(vals.shape[0])
    yes_n = int((vals == 1).sum())
    no_n  = int((vals == 0).sum())
    combined_clean_rows.append({
        'Metric': 'Physical Activity in last 3 months', 'Group': group_name,
        'Total': total, 'Sum': pd.NA, 'Mean': pd.NA, 'SD': pd.NA,
        'Yes_n(%)': f'{yes_n} ({(100*yes_n/total):.1f}%)' if total else '0 (0.0%)',
        'No_n(%)':  f'{no_n} ({(100*no_n/total):.1f}%)'  if total else '0 (0.0%)',
    })

combined_summary_cov_clean = pd.DataFrame(combined_clean_rows)
combined_summary_cov_clean = combined_summary_cov_clean[[
    'Metric', 'Group', 'Total', 'Sum', 'Mean', 'SD', 'Yes_n(%)', 'No_n(%)'
]]

after_path = os.path.join(pl_csv_dir, 'Table1_after_cleaning.csv')
combined_summary_cov_clean.to_csv(after_path, index=False)
print(f'Saved Table1 after cleaning to: {after_path}')
display(combined_summary_cov_clean)


In [ ]:
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

target_col = 'physical_activity'
plot_df = cov_matrix.copy()

target_numeric = pd.to_numeric(plot_df[target_col], errors='coerce')
feature_cols   = [col for col in plot_df.columns if col != target_col]
feature_matrix = plot_df[feature_cols].apply(pd.to_numeric, errors='coerce')
feature_matrix = feature_matrix.fillna(feature_matrix.median(numeric_only=True)).fillna(0)

observed_mask = target_numeric.notna()
missing_mask  = target_numeric.isna()
use_fallback  = int(missing_mask.sum()) == 0

if use_fallback:
    if fallback_samples:
        missing_mask  = plot_df.index.isin(fallback_samples)
        observed_mask = ~missing_mask
        print('No missing values; showing previously imputed samples as missing markers.')
    else:
        print('No missing values and no fallback samples — all points shown as observed.')
else:
    print(f'Found {int(missing_mask.sum())} missing physical_activity value(s).')

scaler = StandardScaler()
scaled_features = scaler.fit_transform(feature_matrix)
pca = PCA(n_components=2, random_state=42)
knn_X_2d = pca.fit_transform(scaled_features)

obs_mask  = np.asarray(observed_mask)
miss_mask = np.asarray(missing_mask)
obs_idx   = np.where(obs_mask)[0]
miss_idx  = np.where(miss_mask)[0]
obs_labels = pd.to_numeric(plot_df.loc[observed_mask, target_col], errors='coerce')
zero_idx = np.where(obs_labels.to_numpy() == 0)[0]
one_idx  = np.where(obs_labels.to_numpy() == 1)[0]

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(knn_X_2d[obs_idx[zero_idx], 0], knn_X_2d[obs_idx[zero_idx], 1],
           c='#1f77b4', s=55, alpha=0.8, label='Observed phys=0')
ax.scatter(knn_X_2d[obs_idx[one_idx],  0], knn_X_2d[obs_idx[one_idx],  1],
           c='#d62728', s=55, alpha=0.8, label='Observed phys=1')
if miss_idx.size > 0:
    ax.scatter(knn_X_2d[miss_idx, 0], knn_X_2d[miss_idx, 1],
               c='black', marker='X', s=140, linewidths=1.5, label='Missing / imputed')
    for sample, miss_pos in zip(plot_df.index[miss_mask], miss_idx):
        ax.annotate(sample, (knn_X_2d[miss_pos, 0], knn_X_2d[miss_pos, 1]),
                    textcoords='offset points', xytext=(0, 12), ha='center', fontsize=8)

ax.set_title('Step 1: Observed and Missing Samples in PCA Space (Placenta)')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.legend(loc='best'); ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(pl_img_dir, 'pca_observed_missing_samples.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
target_col = 'physical_activity'
plot_df = cov_matrix.copy()

target_numeric = pd.to_numeric(plot_df[target_col], errors='coerce')
feature_cols   = [col for col in plot_df.columns if col != target_col]
feature_matrix = plot_df[feature_cols].apply(pd.to_numeric, errors='coerce')
feature_matrix = feature_matrix.fillna(feature_matrix.median(numeric_only=True)).fillna(0)

observed_mask = target_numeric.notna()
missing_mask  = target_numeric.isna()
use_fallback  = int(missing_mask.sum()) == 0

if use_fallback:
    if fallback_samples:
        missing_mask  = plot_df.index.isin(fallback_samples)
        observed_mask = ~missing_mask
        print('Reconstructing neighbour map for previously imputed samples.')
    else:
        print('No missing values and no fallback samples — nothing to map.')
else:
    print(f'Found {int(missing_mask.sum())} missing physical_activity value(s).')

scaler = StandardScaler()
scaled_features = scaler.fit_transform(feature_matrix)
pca = PCA(n_components=2, random_state=42)
knn_X_2d = pca.fit_transform(scaled_features)

obs_mask  = np.asarray(observed_mask)
miss_mask = np.asarray(missing_mask)
obs_idx   = np.where(obs_mask)[0]
miss_idx  = np.where(miss_mask)[0]
obs_labels     = pd.to_numeric(plot_df.loc[observed_mask, target_col], errors='coerce')
observed_index = plot_df.index[observed_mask]
missing_index  = plot_df.index[missing_mask]
zero_idx = np.where(obs_labels.to_numpy() == 0)[0]
one_idx  = np.where(obs_labels.to_numpy() == 1)[0]

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(knn_X_2d[obs_idx[zero_idx], 0], knn_X_2d[obs_idx[zero_idx], 1],
           c='#1f77b4', s=55, alpha=0.8, label='Observed phys=0')
ax.scatter(knn_X_2d[obs_idx[one_idx],  0], knn_X_2d[obs_idx[one_idx],  1],
           c='#d62728', s=55, alpha=0.8, label='Observed phys=1')

if miss_idx.size > 0:
    n_neighbors = min(5, int(obs_mask.sum()))
    nn = NearestNeighbors(n_neighbors=n_neighbors, metric='euclidean')
    nn.fit(scaled_features[obs_mask])
    distances, neighbor_positions = nn.kneighbors(scaled_features[miss_mask])

    ax.scatter(knn_X_2d[miss_idx, 0], knn_X_2d[miss_idx, 1],
               c='black', marker='X', s=140, linewidths=1.5, label='Missing / imputed')

    for i, sample in enumerate(missing_index):
        miss_pos       = miss_idx[i]
        nbr_rel        = neighbor_positions[i]
        nbr_abs        = obs_idx[nbr_rel]
        nbr_samples    = observed_index[nbr_rel].tolist()
        nbr_values     = obs_labels.iloc[nbr_rel].to_numpy()
        nbr_dist       = distances[i]
        weights        = 1.0 / np.maximum(nbr_dist, 1e-9)
        weighted_score = float(np.sum(weights * nbr_values) / np.sum(weights))
        assigned_value = int(round(weighted_score))

        ax.annotate(sample, (knn_X_2d[miss_pos, 0], knn_X_2d[miss_pos, 1]),
                    textcoords='offset points', xytext=(0, 12), ha='center', fontsize=8)
        for nbr_pos, nbr_name in zip(nbr_abs, nbr_samples):
            ax.plot([knn_X_2d[miss_pos, 0], knn_X_2d[nbr_pos, 0]],
                    [knn_X_2d[miss_pos, 1], knn_X_2d[nbr_pos, 1]],
                    color='gray', alpha=0.45, linewidth=1)
            ax.annotate(nbr_name, (knn_X_2d[nbr_pos, 0], knn_X_2d[nbr_pos, 1]),
                        textcoords='offset points', xytext=(4, 4), fontsize=7, color='dimgray')
        ax.annotate(f'->{assigned_value}', (knn_X_2d[miss_pos, 0], knn_X_2d[miss_pos, 1]),
                    textcoords='offset points', xytext=(0, -16), ha='center', fontsize=8, color='black')

ax.set_title('Step 2a: Missing Samples and Their Nearest-Neighbour Mapping (Placenta)')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.legend(loc='best'); ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(pl_img_dir, 'pca_knn_nearest_neighbours.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
cov_clean.head(40)


In [ ]:
placenta_clean = placenta_matrix.loc[cov_clean.index]
print(f'Placenta data shape after matching: {placenta_clean.shape}')


In [ ]:
print('Data types in cleaned covariate matrix:')
print(cov_clean.dtypes)
print('\nFactor covariate levels:')
print('GDM:', cov_clean['GDM'].cat.categories.tolist())
print('physical_activity:', cov_clean['physical_activity'].cat.categories.tolist())


In [ ]:
placenta_clean.head(40)


In [ ]:
# Steps 6-13: Main loop across all placenta metabolites
metabolite_annotations = list(placenta_clean.columns)

p_store         = []
m_GDM_store     = []
m_non_GDM_store = []
log_fc_store    = []
raw_fc_store    = []

print(f'Starting loop over {len(metabolite_annotations)} metabolites...')
print('This will apply Steps 6-13 to each metabolite.\n')

for i, metabolite_name in enumerate(metabolite_annotations, start=1):
    model_data = pd.concat(
        [placenta_clean[metabolite_name].rename('metabolite'), cov_clean], axis=1
    ).dropna()

    if model_data.empty or model_data['GDM'].nunique() < 2:
        p_store.append(np.nan)
        m_GDM_store.append(np.nan)
        m_non_GDM_store.append(np.nan)
        log_fc_store.append(np.nan)
        raw_fc_store.append(np.nan)
    else:
        formula = 'metabolite ~ GDM + ' + ' + '.join(
            [col for col in cov_clean.columns if col != 'GDM']
        )
        fit = sm.formula.ols(formula, data=model_data).fit()

        gdm_pval = (
            fit.pvalues['GDM[T.1]']
            if 'GDM[T.1]' in fit.pvalues.index
            else fit.pvalues.get('GDM', np.nan)
        )

        m_gdm     = model_data.loc[model_data['GDM'] == 1, 'metabolite'].mean()
        m_non_gdm = model_data.loc[model_data['GDM'] == 0, 'metabolite'].mean()

        log_fc = m_gdm - m_non_gdm if pd.notna(m_gdm) and pd.notna(m_non_gdm) else np.nan
        raw_fc = 2 ** log_fc if pd.notna(log_fc) else np.nan

        p_store.append(gdm_pval)
        m_GDM_store.append(m_gdm)
        m_non_GDM_store.append(m_non_gdm)
        log_fc_store.append(log_fc)
        raw_fc_store.append(raw_fc)

    if i % 100 == 0:
        print(f'  Processed {i} metabolites...')

print(f'\nCompleted loop. Processed {len(metabolite_annotations)} metabolites.')
print(f'  p-values stored: {len(p_store)}')
print(f'  GDM means stored: {len(m_GDM_store)}')
print(f'  Non-GDM means stored: {len(m_non_GDM_store)}')
print(f'  Fold changes stored: {len(raw_fc_store)}')


In [ ]:
p_array         = np.array(p_store)
m_gdm_array     = np.array(m_GDM_store)
m_non_gdm_array = np.array(m_non_GDM_store)
log_fc_array    = np.array(log_fc_store)
raw_fc_array    = np.array(raw_fc_store)

print('Results vectors prepared:')
print(f'  p-values: {len(p_array)}')
print(f'  GDM means: {len(m_gdm_array)}')
print(f'  Non-GDM means: {len(m_non_gdm_array)}')
print(f'  Log fold changes: {len(log_fc_array)}')
print(f'  Raw fold changes: {len(raw_fc_array)}')

print(f'\nP-value summary:')
print(f'  Min: {np.nanmin(p_array)}')
print(f'  Max: {np.nanmax(p_array)}')
print(f'  Mean: {np.nanmean(p_array)}')
print(f'  Median: {np.nanmedian(p_array)}')


In [ ]:
from statsmodels.stats.multitest import multipletests

valid_mask = ~np.isnan(p_array)
pvals_adj_bh = np.full(len(p_array), np.nan)

if valid_mask.any():
    _, pvals_adj_bh[valid_mask], _, _ = multipletests(p_array[valid_mask], method='fdr_bh')

reject_bh = ~np.isnan(pvals_adj_bh) & (pvals_adj_bh < 0.05)

print('Benjamini-Hochberg Correction Results:')
print(f'  Number of significant metabolites (FDR < 0.05): {reject_bh.sum()}')
print(f'  Min adjusted p-value: {np.nanmin(pvals_adj_bh)}')
print(f'  Max adjusted p-value: {np.nanmax(pvals_adj_bh)}')
print(f'  Mean adjusted p-value: {np.nanmean(pvals_adj_bh)}')
print(f'  Median adjusted p-value: {np.nanmedian(pvals_adj_bh)}')


In [ ]:
try:
    from qvalue import qvalue
    q_vals = qvalue(p_array)['qvalues']
    print('Q-values calculated using qvalue package')
except ImportError:
    print('qvalue package not available, using BH adjusted p-values as approximation')
    q_vals = pvals_adj_bh

print(f'\nQ-value Summary:')
print(f'  Min: {np.nanmin(q_vals)}')
print(f'  Max: {np.nanmax(q_vals)}')
print(f'  Mean: {np.nanmean(q_vals)}')
print(f'  Median: {np.nanmedian(q_vals)}')


In [ ]:
if 'metabolite_names' in globals():
    metabolite_names_local = metabolite_names
else:
    metabolite_names_local = metabolite_annotations

if PLACENTA_ANNO is not None and os.path.exists(PLACENTA_ANNO):
    anno_df = pd.read_csv(PLACENTA_ANNO, index_col=0)
else:
    anno_df = None

results_df = pd.DataFrame({
    'Metabolite': metabolite_names_local,
    'p_value_raw': p_array,
    'p_value_corrected': pvals_adj_bh,
    'q_value': q_vals,
    'Mean_GDM': m_gdm_array,
    'Mean_Non_GDM': m_non_gdm_array,
    'Log_Fold_Change': log_fc_array,
    'Raw_Fold_Change': raw_fc_array,
})

if anno_df is not None:
    anno_mapping = anno_df.to_dict('index')
    annotation_cols = [
        anno_mapping.get(met, {col: None for col in anno_df.columns})
        for met in metabolite_names_local
    ]
    anno_expanded = pd.DataFrame(annotation_cols)
    results_df = pd.concat([results_df, anno_expanded], axis=1)

results_df = results_df.sort_values('p_value_raw')

print('Results Table Summary:')
print(f'  Total metabolites: {len(results_df)}')
print(f'\nFirst 10 rows (sorted by p-value):')
results_df.head(10)


In [ ]:
save_path = os.path.join(pl_csv_dir, 'Linear_Model_data.csv')
results_df.to_csv(save_path, index=False)
print(f'Saved results to: {save_path}')
print(f'Rows: {len(results_df)}, Columns: {results_df.shape[1]}')


In [ ]:
alpha = 0.05
cmp_df = results_df[['p_value_raw', 'p_value_corrected']].dropna().copy()

n_total_cmp = len(cmp_df)
raw_yes = int((cmp_df['p_value_raw']       < alpha).sum())
raw_no  = n_total_cmp - raw_yes
fdr_yes = int((cmp_df['p_value_corrected'] < alpha).sum())
fdr_no  = n_total_cmp - fdr_yes

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))

bars1 = axes[0].bar(['NO', 'YES'], [raw_no, raw_yes], color=['#de0848', '#04066c'], width=0.6)
axes[0].set_title(f'Raw p < {alpha}')
axes[0].set_ylabel('Number of metabolites')
axes[0].grid(axis='y', alpha=0.25)
for b in bars1:
    axes[0].text(b.get_x() + b.get_width()/2, b.get_height() + max(1, n_total_cmp*0.01),
                 f'{int(b.get_height())}', ha='center', va='bottom', fontsize=10)

bars2 = axes[1].bar(['NO', 'YES'], [fdr_no, fdr_yes], color=['#de0848', '#a018d2'], width=0.6)
axes[1].set_title(f'FDR < {alpha}')
axes[1].grid(axis='y', alpha=0.25)
for b in bars2:
    axes[1].text(b.get_x() + b.get_width()/2, b.get_height() + max(1, n_total_cmp*0.01),
                 f'{int(b.get_height())}', ha='center', va='bottom', fontsize=10)

fig.suptitle('Significance Comparison (Placenta): Raw vs Multiple-Testing Corrected', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(pl_img_dir, 'significance_comparison_alpha_0_05.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'Raw p<{alpha}: {raw_yes}/{n_total_cmp}')
print(f'FDR<{alpha}: {fdr_yes}/{n_total_cmp}')


In [ ]:
metabolites_p05 = results_df[
    results_df['p_value_raw'] < 0.05
][['Metabolite', 'p_value_raw', 'p_value_corrected', 'Log_Fold_Change',
   'Mean_GDM', 'Mean_Non_GDM']].reset_index(drop=True)

print(f"{'='*100}")
print(f'SIGNIFICANT METABOLITES (Raw p < 0.05): {len(metabolites_p05)} metabolites')
print(f"{'='*100}\n")

for idx, row in metabolites_p05.iterrows():
    print(f"{idx+1:3d}. {str(row['Metabolite']):>10s}  |  "
          f"Raw p: {row['p_value_raw']:.6f}  |  "
          f"FDR: {row['p_value_corrected']:.4f}  |  "
          f"Log FC: {row['Log_Fold_Change']:>7.4f}  |  "
          f"Mean GDM: {row['Mean_GDM']:.4f}  |  Mean Non-GDM: {row['Mean_Non_GDM']:.4f}")

p05_path = os.path.join(pl_csv_dir, 'metabolites_raw_p_lt_0_05.csv')
metabolites_p05.to_csv(p05_path, index=False)
print(f'\nSaved to: {p05_path}')


In [ ]:
alpha = 0.1
cmp_df = results_df[['p_value_raw', 'p_value_corrected']].dropna().copy()

n_total_cmp = len(cmp_df)
raw_yes = int((cmp_df['p_value_raw']       < alpha).sum())
raw_no  = n_total_cmp - raw_yes
fdr_yes = int((cmp_df['p_value_corrected'] < alpha).sum())
fdr_no  = n_total_cmp - fdr_yes

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))

bars1 = axes[0].bar(['NO', 'YES'], [raw_no, raw_yes], color=['#de0848', '#04066c'], width=0.6)
axes[0].set_title(f'Raw p < {alpha}')
axes[0].set_ylabel('Number of metabolites')
axes[0].grid(axis='y', alpha=0.25)
for b in bars1:
    axes[0].text(b.get_x() + b.get_width()/2, b.get_height() + max(1, n_total_cmp*0.01),
                 f'{int(b.get_height())}', ha='center', va='bottom', fontsize=10)

bars2 = axes[1].bar(['NO', 'YES'], [fdr_no, fdr_yes], color=['#de0848', '#a018d2'], width=0.6)
axes[1].set_title(f'FDR < {alpha}')
axes[1].grid(axis='y', alpha=0.25)
for b in bars2:
    axes[1].text(b.get_x() + b.get_width()/2, b.get_height() + max(1, n_total_cmp*0.01),
                 f'{int(b.get_height())}', ha='center', va='bottom', fontsize=10)

fig.suptitle('Significance Comparison alpha=0.1 (Placenta): Raw vs Multiple-Testing Corrected', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(pl_img_dir, 'significance_comparison_alpha_0_10.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'Raw p<{alpha}: {raw_yes}/{n_total_cmp}')
print(f'FDR<{alpha}: {fdr_yes}/{n_total_cmp}')


In [ ]:
metabolites_p10 = results_df[
    results_df['p_value_raw'] < 0.1
][['Metabolite', 'p_value_raw', 'p_value_corrected', 'Log_Fold_Change',
   'Mean_GDM', 'Mean_Non_GDM']].reset_index(drop=True)

print(f"{'='*100}")
print(f'SIGNIFICANT METABOLITES (Raw p < 0.1): {len(metabolites_p10)} metabolites')
print(f"{'='*100}\n")

for idx, row in metabolites_p10.iterrows():
    print(f"{idx+1:3d}. {str(row['Metabolite']):>10s}  |  "
          f"Raw p: {row['p_value_raw']:.6f}  |  "
          f"FDR: {row['p_value_corrected']:.4f}  |  "
          f"Log FC: {row['Log_Fold_Change']:>7.4f}  |  "
          f"Mean GDM: {row['Mean_GDM']:.4f}  |  Mean Non-GDM: {row['Mean_Non_GDM']:.4f}")

p10_path = os.path.join(pl_csv_dir, 'metabolites_raw_p_lt_0_1_detailed.csv')
metabolites_p10.to_csv(p10_path, index=False)
print(f'\nSaved to: {p10_path}')


In [ ]:
plot_df2 = results_df.dropna(subset=['Log_Fold_Change', 'p_value_corrected']).copy()
plot_df2['neg_log10_fdr'] = -np.log10(np.clip(plot_df2['p_value_corrected'].astype(float), 1e-300, 1.0))
plot_df2['significant_fdr'] = plot_df2['p_value_corrected'].astype(float) < 0.05

sig_n     = int(plot_df2['significant_fdr'].sum())
non_sig_n = int((~plot_df2['significant_fdr']).sum())
total_n   = sig_n + non_sig_n
sig_pct   = (100 * sig_n / total_n) if total_n else 0.0

fig, ax = plt.subplots(figsize=(6, 5))
ax.bar(['Significant', 'Not significant'], [sig_n, non_sig_n], color=['#d1495b', '#9aa0a6'])
ax.set_ylabel('Number of metabolites')
ax.set_title(f'FDR Summary — Placenta (FDR < 0.05: {sig_n}/{total_n}, {sig_pct:.1f}%)')
for i, v in enumerate([sig_n, non_sig_n]):
    ax.text(i, v + max(1, total_n * 0.01), str(v), ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(pl_img_dir, 'fdr_summary_bar_plot.png'), dpi=150, bbox_inches='tight')
plt.show()

if sig_n > 0:
    print(f'Conclusion: {sig_n} placenta metabolites are significantly different between GDM-positive and non-GDM women at FDR < 0.05.')
else:
    print('Conclusion: No statistically significant placenta metabolite differences detected between GDM-positive and non-GDM women at FDR < 0.05.')


In [ ]:
placenta_clean_export = os.path.join(pl_csv_dir, 'placenta_clean_from_python.csv')
cov_clean_export      = os.path.join(pl_csv_dir, 'cov_clean_from_python.csv')

placenta_clean.to_csv(placenta_clean_export, index=True, index_label='Sample')
cov_clean.to_csv(cov_clean_export,           index=True, index_label='Sample')

print(f'Saved placenta_clean to: {placenta_clean_export}')
print(f'Saved cov_clean to:      {cov_clean_export}')
print(f'placenta_clean shape: {placenta_clean.shape}')
print(f'cov_clean shape:      {cov_clean.shape}')


In [ ]:
metabolites_raw_p_01 = results_df[results_df['p_value_raw'] < 0.1]['Metabolite'].tolist()

txt_path = os.path.join(pl_csv_dir, 'metabolites_raw_p_lt_0_1.txt')
with open(txt_path, 'w') as f:
    for metabolite in metabolites_raw_p_01:
        f.write(f'{metabolite}\n')

print(f'Saved {len(metabolites_raw_p_01)} metabolite names to: {txt_path}')
print('\nFirst 10 metabolite names:')
for i, met in enumerate(metabolites_raw_p_01[:10], 1):
    print(f'  {i}. {met}')


In [ ]:
metabolites_df = results_df[
    results_df['p_value_raw'] < 0.1
][['Metabolite', 'p_value_raw', 'p_value_corrected', 'Log_Fold_Change']].reset_index(drop=True)
metabolites_df.index.name = 'Index'
metabolites_df = metabolites_df.reset_index()

csv_path = os.path.join(pl_csv_dir, 'metabolites_raw_p_lt_0_1.csv')
metabolites_df.to_csv(csv_path, index=False)

print(f'Saved metabolites (raw p < 0.1) table to: {csv_path}')
print(f'Table shape: {metabolites_df.shape}')
print('\nFirst 10 rows:')
print(metabolites_df.head(10))
